In [6]:
import pandas as pd
import requests
import json
import re
from datetime import datetime
from typing import Optional, Dict, Any, List
from dateutil import parser as date_parser
import time
import itertools

In [7]:
# === НАСТРОЙКИ ===
BASE_URL = 'http://v-ahunter-lb-pb.vsk.ru'
USER_ID = 'aRTDM'

URLS = {
    'person': f"{BASE_URL}/site/cleanse/person",
    'phone': f"{BASE_URL}/site/cleanse/phone",
    'address': f"{BASE_URL}/site/cleanse/address",
    'record': f"{BASE_URL}/site/cleanse/record"
}

INPUT_FILE = 'СВОД.xlsx'
PREPROCESSED_FILE = 'СВОД_preprocessed.xlsx'
OUTPUT_FILE = 'СВОД_cleaned.xlsx'

COLUMNS_TO_PROCESS = {
    'ФИО': 'person',
    'ТЕЛЕФОН': 'phone',
    'АДРЕС': 'address'
}

In [8]:
# ============================================================================
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ============================================================================

def convert_to_string(value) -> str:
    """
    Конвертирует значение в строку, обрабатывая float, int и другие типы.
    """
    if value is None or pd.isna(value):
        return ''
    
    # Если это float (например, 2901135536.0)
    if isinstance(value, float):
        # Проверяем, целое ли это число
        if value.is_integer():
            return str(int(value))
        else:
            return str(value)
    
    # Если это int
    if isinstance(value, int):
        return str(value)
    
    # Если это datetime
    if isinstance(value, datetime):
        return value.strftime('%d.%m.%Y')
    
    # Иначе просто конвертируем в строку
    return str(value).strip()


# ============================================================================
# ФУНКЦИИ ОЧИСТКИ И НОРМАЛИЗАЦИИ
# ============================================================================

def clean_fio(fio_text: str) -> str:
    """Очищает ФИО от лишних приписок"""
    if not fio_text or pd.isna(fio_text):
        return fio_text
    
    text = convert_to_string(fio_text)
    
    # Сохраняем префикс "ИП " если есть
    ip_prefix = ""
    if text.upper().startswith("ИП "):
        ip_prefix = "ИП "
        text = text[3:].strip()
    elif text.upper().startswith("ИНДИВИДУАЛЬНЫЙ ПРЕДПРИНИМАТЕЛЬ "):
        ip_prefix = "ИП "
        text = re.sub(r'^ИНДИВИДУАЛЬНЫЙ\s+ПРЕДПРИНИМАТЕЛЬ\s+', '', text, flags=re.IGNORECASE).strip()
    
    # Убираем текст в скобках
    text = re.sub(r'\s*\([^)]*\)', '', text)
    
    # Убираем текст после запятой (если это не часть ФИО)
    parts = text.split(',')
    if len(parts) > 1:
        first_part = parts[0].strip()
        words = first_part.split()
        if 2 <= len(words) <= 3:
            text = first_part
    
    # Убираем паспортные данные
    text = re.sub(r'\s*паспорт\s+серия\s*\d+\s+номер\s*\d+', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*паспорт\s+\d{4}\s+\d{6}', '', text, flags=re.IGNORECASE)
    
    # Убираем слова "автоюрист", "адвокат" и т.д. в конце
    text = re.sub(r'\s*,?\s*(автоюрист|адвокат|директор|руководитель|менеджер|владелец)\b.*$', '', text, flags=re.IGNORECASE)
    
    # Очищаем лишние пробелы
    text = ' '.join(text.split())
    
    return ip_prefix + text if text else convert_to_string(fio_text)


def clean_company(company_text: str) -> str:
    """
    Очищает название компании от лишних данных.
    Приводит к формату: ТИП "НАЗВАНИЕ" (в верхнем регистре)
    """
    if not company_text or pd.isna(company_text):
        return company_text
    
    text = convert_to_string(company_text)
    
    # Список типов юрлиц
    org_types = r'(ООО|ИП|АО|ЗАО|ОАО|ПАО|МУП|ГУП|НКО|НАО|ФГУП|ФГБУ|ГБУ|ГКУ|МБУ|МКУ)'
    
    # Паттерн 1: ТИП "НАЗВАНИЕ" или ТИП 'НАЗВАНИЕ'
    pattern1 = rf'{org_types}\s*["\']([^"\']+)["\']'
    match = re.search(pattern1, text, flags=re.IGNORECASE)
    if match:
        org_type = match.group(1).upper()
        org_name = match.group(2).strip().upper()
        return f'{org_type} "{org_name}"'
    
    # Паттерн 2: ТИП НАЗВАНИЕ (без кавычек, до запятой или конца строки)
    pattern2 = rf'{org_types}\s+([^\s,\(\);]+(?:\s+[^\s,\(\);]+)*)'
    match = re.search(pattern2, text, flags=re.IGNORECASE)
    if match:
        org_type = match.group(1).upper()
        org_name = match.group(2).strip()
        # Убираем лишние слова в конце (директор, ИНН и т.д.)
        org_name = re.sub(r'\s*-\s*директор.*$', '', org_name, flags=re.IGNORECASE)
        org_name = re.sub(r'\s*,\s*директор.*$', '', org_name, flags=re.IGNORECASE)
        org_name = re.sub(r'\s*\(ИНН.*$', '', org_name, flags=re.IGNORECASE)
        org_name = re.sub(r'\s*,\s*автоюрист.*$', '', org_name, flags=re.IGNORECASE)
        org_name = org_name.strip().upper()
        return f'{org_type} "{org_name}"'
    
    # Паттерн 3: Просто ТИП НАЗВАНИЕ (одно слово)
    pattern3 = rf'{org_types}\s+(\S+)'
    match = re.search(pattern3, text, flags=re.IGNORECASE)
    if match:
        org_type = match.group(1).upper()
        org_name = match.group(2).strip().upper()
        return f'{org_type} "{org_name}"'
    
    # Если не нашли паттерн, пробуем убрать лишнее
    text = re.sub(r'\s*\([^)]*ИНН[^)]*\)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*,\s*директор\s+.*$', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*-\s*директор\s+.*$', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*,\s*автоюрист.*$', '', text, flags=re.IGNORECASE)
    
    # Приводим к верхнему регистру
    text = text.strip().upper()
    
    return text


def clean_inn(inn_text) -> str:
    """
    Очищает ИНН от слова 'ИНН' и пробелов.
    Возвращает строку с цифрами.
    """
    if not inn_text or pd.isna(inn_text):
        return convert_to_string(inn_text)
    
    text = convert_to_string(inn_text)
    text = re.sub(r'\bИНН\b', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', '', text)
    
    return text


def split_inn_values(inn_text) -> List[str]:
    """
    Разделяет ИНН по паттерну "ИНН" перед цифрами.
    Пример: "ИНН 2901135536 ИНН 2901204758" -> ["2901135536", "2901204758"]
    """
    if not inn_text or pd.isna(inn_text):
        return [convert_to_string(inn_text)]
    
    text = convert_to_string(inn_text)
    
    # Разделяем по паттерну "ИНН" (с пробелами вокруг)
    parts = re.split(r'\s*ИНН\s*', text, flags=re.IGNORECASE)
    
    # Очищаем каждую часть от пробелов
    values = [p.strip() for p in parts if p.strip()]
    
    return values if values else [convert_to_string(inn_text)]


def split_fio_or_company_values(text: str) -> List[str]:
    """
    Умное разделение ФИО или компаний.
    Разделяет только если после разделителя идет новое ФИО (2-3 слова) или тип юрлица.
    """
    if not text or pd.isna(text):
        return [convert_to_string(text)]
    
    text = convert_to_string(text)
    
    # Разделители: слэш, перенос строки, точка с запятой
    # Запятая - только если после нее идет тип юрлица или новое ФИО
    
    # Сначала разделяем по слэшу и переносу строки
    parts = re.split(r'\s*[/;]\s*|\n|\r', text)
    
    result = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        
        # Проверяем, есть ли внутри запятые, после которых идет тип юрлица
        # Паттерн: что-то, ООО/ИП/АО...
        org_pattern = r'(ООО|ИП|АО|ЗАО|ОАО|ПАО|МУП|ГУП|НКО)\s'
        
        if re.search(org_pattern, part, flags=re.IGNORECASE):
            # Разделяем по запятой, если после нее идет тип юрлица
            sub_parts = re.split(r'\s*,\s*(?=(?:ООО|ИП|АО|ЗАО|ОАО|ПАО|МУП|ГУП|НКО)\s)', part, flags=re.IGNORECASE)
            result.extend([p.strip() for p in sub_parts if p.strip()])
        else:
            result.append(part)
    
    return result if result else [convert_to_string(text)]


def split_phone_or_email_values(text: str) -> List[str]:
    """
    Разделение телефонов и email.
    Разделители: запятая, точка с запятой, слэш, перенос строки.
    """
    if not text or pd.isna(text):
        return [convert_to_string(text)]
    
    text = convert_to_string(text)
    
    # Заменяем все разделители на маркер
    text = re.sub(r'[\n\r]+', '|||', text)
    text = re.sub(r'\s*;\s*', '|||', text)
    text = re.sub(r'\s*/\s*', '|||', text)
    text = re.sub(r'\s*,\s*', '|||', text)
    
    values = [v.strip() for v in text.split('|||') if v.strip()]
    
    return values if values else [convert_to_string(text)]


def split_multiple_values(cell_value, column_type: str) -> List[str]:
    """
    Разделяет ячейку с несколькими значениями на список.
    column_type: 'fio', 'inn', 'phone_email', 'other'
    """
    if not cell_value or pd.isna(cell_value):
        return [convert_to_string(cell_value)]
    
    if column_type == 'inn':
        return split_inn_values(cell_value)
    elif column_type == 'fio_company':
        return split_fio_or_company_values(cell_value)
    elif column_type == 'phone_email':
        return split_phone_or_email_values(cell_value)
    else:
        # Для остальных - простое разделение
        text = convert_to_string(cell_value)
        text = re.sub(r'[\n\r]+', '|||', text)
        text = re.sub(r'\s*;\s*', '|||', text)
        text = re.sub(r'\s*/\s*', '|||', text)
        values = [v.strip() for v in text.split('|||') if v.strip()]
        return values if values else [convert_to_string(cell_value)]


def expand_row(row: pd.Series) -> List[pd.Series]:
    """
    Разделяет одну строку на несколько, если есть множественные значения.
    АДРЕС не участвует в разделении!
    """
    multi_values = {}
    
    # ФИО - умное разделение
    if 'ФИО' in row.index:
        val = row['ФИО']
        if not pd.isna(val) and convert_to_string(val).strip():
            split_vals = split_multiple_values(val, 'fio_company')
            if len(split_vals) > 1:
                multi_values['ФИО'] = split_vals
    
    # ТЕЛЕФОН - простое разделение
    if 'ТЕЛЕФОН' in row.index:
        val = row['ТЕЛЕФОН']
        if not pd.isna(val) and convert_to_string(val).strip():
            split_vals = split_multiple_values(val, 'phone_email')
            if len(split_vals) > 1:
                multi_values['ТЕЛЕФОН'] = split_vals
    
    # E-MAIL - простое разделение
    if 'E-MAIL' in row.index:
        val = row['E-MAIL']
        if not pd.isna(val) and convert_to_string(val).strip():
            split_vals = split_multiple_values(val, 'phone_email')
            if len(split_vals) > 1:
                multi_values['E-MAIL'] = split_vals
    
    # ИНН - разделение по паттерну "ИНН"
    if 'ИНН' in row.index:
        val = row['ИНН']
        if not pd.isna(val) and convert_to_string(val).strip():
            split_vals = split_multiple_values(val, 'inn')
            if len(split_vals) > 1:
                multi_values['ИНН'] = split_vals
    
    # ДАТА РОЖДЕНИЯ - простое разделение
    if 'ДАТА РОЖДЕНИЯ' in row.index:
        val = row['ДАТА РОЖДЕНИЯ']
        if not pd.isna(val) and convert_to_string(val).strip():
            split_vals = split_multiple_values(val, 'other')
            if len(split_vals) > 1:
                multi_values['ДАТА РОЖДЕНИЯ'] = split_vals
    
    # АДРЕС НЕ РАЗДЕЛЯЕМ!
    
    # Если нет множественных значений, возвращаем исходную строку
    if not multi_values:
        return [row]
    
    # Создаем все комбинации
    columns = list(multi_values.keys())
    value_lists = [multi_values[col] for col in columns]
    combinations = list(itertools.product(*value_lists))
    
    expanded_rows = []
    for combo in combinations:
        new_row = row.copy()
        for i, col in enumerate(columns):
            new_row[col] = combo[i]
        expanded_rows.append(new_row)
    
    return expanded_rows


def normalize_email(email: str) -> str:
    """Нормализует email"""
    if not email or pd.isna(email):
        return email
    
    original = convert_to_string(email)
    email = original.strip().lower()
    email = re.sub(r'\s+', '', email)
    
    email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
    
    if re.match(email_pattern, email):
        return email
    else:
        return original


def normalize_date(date_value) -> str:
    """Приводит дату к формату дд.мм.гггг"""
    if not date_value or pd.isna(date_value):
        return convert_to_string(date_value)
    
    if isinstance(date_value, datetime):
        return date_value.strftime('%d.%m.%Y')
    
    original = convert_to_string(date_value)
    
    try:
        date_str = original.strip()
        parsed_date = date_parser.parse(date_str, dayfirst=True)
        return parsed_date.strftime('%d.%m.%Y')
    except (ValueError, TypeError, OverflowError):
        return original


def normalize_inn(inn_value) -> str:
    """Нормализует ИНН"""
    if not inn_value or pd.isna(inn_value):
        return convert_to_string(inn_value)
    
    original = convert_to_string(inn_value)
    cleaned = re.sub(r'\s+', '', original)
    
    if cleaned.isdigit():
        return cleaned
    else:
        return original


# ============================================================================
# ФУНКЦИЯ 1: ПРЕДВАРИТЕЛЬНАЯ ОБРАБОТКА
# ============================================================================

def preprocess_excel():
    """
    Предварительная обработка данных:
    - Очистка ФИО, ИНН, компаний
    - Разделение множественных значений
    - Нормализация email, дат, ИНН
    
    Сохраняет результат в СВОД_preprocessed.xlsx
    """
    print("="*80)
    print("ПРЕДВАРИТЕЛЬНАЯ ОБРАБОТКА ДАННЫХ")
    print("="*80)
    
    # Загрузка файла
    print(f"\n1. Загрузка файла: {INPUT_FILE}")
    try:
        df = pd.read_excel(INPUT_FILE)
    except FileNotFoundError:
        print(f"✗ Файл {INPUT_FILE} не найден!")
        return None
    except Exception as e:
        print(f"✗ Ошибка при чтении Excel: {e}")
        return None
    
    print(f"✓ Загружено строк: {len(df)}")
    print(f"✓ Колонки: {list(df.columns)}")
    
    original_count = len(df)
    
    # Шаг 1: Предварительная очистка
    print("\n2. Очистка данных...")
    
    if 'ФИО' in df.columns:
        df['ФИО'] = df['ФИО'].apply(clean_fio)
        print("  ✓ Очищены ФИО")
    
    if 'ИНН' in df.columns:
        df['ИНН'] = df['ИНН'].apply(clean_inn)
        print("  ✓ Очищены ИНН (убрано слово 'ИНН')")
    
    # Шаг 2: Разделение строк с множественными значениями
    print("\n3. Разделение строк с множественными значениями...")
    
    expanded_rows = []
    for idx, row in df.iterrows():
        expanded = expand_row(row)
        expanded_rows.extend(expanded)
    
    df = pd.DataFrame(expanded_rows, columns=df.columns)
    print(f"  ✓ Было строк: {original_count}, стало: {len(df)}")
    
    # Шаг 3: Локальная нормализация
    print("\n4. Локальная нормализация...")
    
    if 'E-MAIL' in df.columns:
        df['E-MAIL'] = df['E-MAIL'].apply(normalize_email)
        print("  ✓ Нормализованы email")
    
    if 'ДАТА РОЖДЕНИЯ' in df.columns:
        df['ДАТА РОЖДЕНИЯ'] = df['ДАТА РОЖДЕНИЯ'].apply(normalize_date)
        print("  ✓ Нормализованы даты рождения")
    
    if 'ИНН' in df.columns:
        df['ИНН'] = df['ИНН'].apply(normalize_inn)
        print("  ✓ Нормализованы ИНН (убраны пробелы)")
    
    # Сохранение результата
    print(f"\n5. Сохранение предварительной таблицы: {PREPROCESSED_FILE}")
    try:
        df.to_excel(PREPROCESSED_FILE, index=False)
        print(f"✓ Готово!")
        print(f"  Строк в предварительной таблице: {len(df)}")
        print(f"  Файл сохранен: {PREPROCESSED_FILE}")
    except Exception as e:
        print(f"✗ Ошибка при сохранении: {e}")
        return None
    
    print("\n" + "="*80)
    print("ПРЕДВАРИТЕЛЬНАЯ ОБРАБОТКА ЗАВЕРШЕНА")
    print("="*80)
    print(f"\nПроверьте файл {PREPROCESSED_FILE}")
    print("Если всё корректно, запустите process_with_ahunter() для обработки через API")
    
    return df


# ============================================================================
# ФУНКЦИЯ 2: ОБРАБОТКА ЧЕРЕЗ AHUNTER API
# ============================================================================

def standardize_record_api(fio: str, phone: str, address: str) -> Dict[str, Any]:
    """Отправляет комбинированный запрос в AHunter API"""
    url = URLS.get('record')
    if not url:
        return None
    
    items = []
    if fio:
        items.append({"type": "person", "role": "ФИО", "body": fio})
    if phone:
        items.append({"type": "phone", "role": "Телефон", "body": phone})
    if address:
        items.append({"type": "address", "role": "Адрес", "body": address})
    
    if not items:
        return None
    
    record_data = {"id": "1", "items": items}
    
    params = {
        'user': USER_ID,
        'output': 'json',
        'input': 'json',
        'query': json.dumps(record_data, ensure_ascii=False)
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if 'errors' in data:
            return None
        
        return data
    except Exception as e:
        return None


def extract_standardized_data(api_response: Dict[str, Any], column_type: str) -> Optional[str]:
    """Извлекает стандартизованное значение из ответа API"""
    if not api_response or 'items' not in api_response:
        return None
    
    for item in api_response.get('items', []):
        if item.get('type') != column_type:
            continue
        
        result = item.get('result', {})
        
        if column_type == 'person':
            persons = result.get('persons', [])
            if persons:
                return persons[0].get('full_name')
        
        elif column_type == 'phone':
            phones = result.get('phones', [])
            if phones:
                best = phones[0]
                zone = best.get('zone_of_range', '')
                number = best.get('canon_phone_number', '')
                
                if zone and number and len(number) == 7:
                    formatted_number = f"{number[:3]}-{number[3:5]}-{number[5:7]}"
                    return f"+7 ({zone}) {formatted_number}"
                elif zone and number:
                    return f"+7 ({zone}) {number}"
                elif number:
                    return number
        
        elif column_type == 'address':
            addresses = result.get('addresses', [])
            if addresses:
                pretty_address = addresses[0].get('pretty')
                if pretty_address:
                    return pretty_address
                
                fields = addresses[0].get('fields', [])
                addr_parts = [f.get('name') for f in fields if f.get('name')]
                if addr_parts:
                    return ', '.join(addr_parts)
    
    return None


def is_legal_entity(value: str) -> bool:
  """
  Проверяет, является ли значение юридическим лицом.
  """
  if not value or pd.isna(value):
    return False
  
  text = convert_to_string(value).strip().upper()
  
  # Паттерны юридических лиц
  legal_patterns = [
    r'^(ООО|ИП|АО|ЗАО|ОАО|ПАО|МУП|ГУП|НКО|НАО|ФГУП|ФГБУ|ГБУ|ГКУ|МБУ|МКУ)\s',
    r'^(ООО|ИП|АО|ЗАО|ОАО|ПАО|МУП|ГУП|НКО|НАО|ФГУП|ФГБУ|ГБУ|ГКУ|МБУ|МКУ)$',
  ]
  
  for pattern in legal_patterns:
    if re.match(pattern, text):
      return True
  
  return False


def process_with_ahunter():
  """
  Обработка предварительно подготовленной таблицы через AHunter API.
  Загружает СВОД_preprocessed.xlsx и сохраняет результат в СВОД_cleaned.xlsx
  Юридические лица в колонке ФИО НЕ отправляются в API, но телефон и адрес отправляются.
  """
  print("="*80)
  print("ОБРАБОТКА ЧЕРЕЗ AHUNTER API")
  print("="*80)
  
  # Загрузка предобработанного файла
  print(f"\n1. Загрузка предварительной таблицы: {PREPROCESSED_FILE}")
  try:
    df = pd.read_excel(PREPROCESSED_FILE)
  except FileNotFoundError:
    print(f"✗ Файл {PREPROCESSED_FILE} не найден!")
    print(" Сначала запустите preprocess_excel() для создания предварительной таблицы")
    return None
  except Exception as e:
    print(f"✗ Ошибка при чтении Excel: {e}")
    return None
  
  print(f"✓ Загружено строк: {len(df)}")
  print(f"✓ Колонки: {list(df.columns)}")
  
  # Обработка через API
  print("\n2. Обработка через AHunter API...")
  processed_count = 0
  skipped_fio_count = 0
  error_count = 0
  
  for idx, row in df.iterrows():
    if idx % 10 == 0:
      print(f" Обработка строки {idx + 1} из {len(df)}...")
    
    # Собираем значения для API
    values = {}
    for col_name in COLUMNS_TO_PROCESS.keys():
      if col_name in df.columns:
        val = row[col_name]
        if pd.isna(val) or not convert_to_string(val).strip():
          values[col_name] = None
        else:
          values[col_name] = convert_to_string(val).strip()
    
    # Если все поля пустые - пропускаем
    if not any(values.values()):
      continue
    
    # Формируем items для API, исключая юрлица из ФИО
    items = []
    
    # ФИО - только если это не юридическое лицо
    fio_value = values.get('ФИО')
    if fio_value:
      if is_legal_entity(fio_value):
        # Это юрлицо - не добавляем в items
        skipped_fio_count += 1
      else:
        # Это физлицо - добавляем в items
        items.append({"type": "person", "role": "ФИО", "body": fio_value})
    
    # Телефон - добавляем всегда если есть
    phone_value = values.get('ТЕЛЕФОН')
    if phone_value:
      items.append({"type": "phone", "role": "Телефон", "body": phone_value})
    
    # Адрес - добавляем всегда если есть
    address_value = values.get('АДРЕС')
    if address_value:
      items.append({"type": "address", "role": "Адрес", "body": address_value})
    
    # Если нет items для отправки - пропускаем
    if not items:
      continue
    
    # Отправляем запрос в API
    api_response = standardize_record_api_from_items(items)
    
    # Обновляем значения
    if api_response:
      for col_name, col_type in COLUMNS_TO_PROCESS.items():
        if col_name not in df.columns:
          continue
        
        # Для ФИО проверяем, не юрлицо ли это
        if col_name == 'ФИО':
          original_value = values.get('ФИО')
          if original_value and is_legal_entity(original_value):
            # Оставляем юрлицо как есть
            continue
        
        original_value = values.get(col_name)
        if not original_value:
          continue
        
        standardized = extract_standardized_data(api_response, col_type)
        if standardized and str(standardized).strip():
          df.at[idx, col_name] = standardized
          processed_count += 1
    else:
      error_count += 1
    
    time.sleep(0.1)
  
  # Сохранение результата
  print(f"\n3. Сохранение финальной таблицы: {OUTPUT_FILE}")
  try:
    df.to_excel(OUTPUT_FILE, index=False)
    print(f"✓ Готово!")
    print(f" Всего строк: {len(df)}")
    print(f" Успешно стандартизировано полей: {processed_count}")
    print(f" Пропущено ФИО (юрлица): {skipped_fio_count}")
    print(f" Ошибок API: {error_count}")
    print(f" Файл сохранен: {OUTPUT_FILE}")
  except Exception as e:
    print(f"✗ Ошибка при сохранении: {e}")
    return None
  
  print("\n" + "="*80)
  print("ОБРАБОТКА ЧЕРЕЗ API ЗАВЕРШЕНА")
  print("="*80)


def standardize_record_api_from_items(items: List[Dict]) -> Dict[str, Any]:
    """
    Отправляет запрос в AHunter API с готовым списком items.
    """
    url = URLS.get('record')
    if not url:
      return None
    
    if not items:
      return None
    
    record_data = {"id": "1", "items": items}
    
    params = {
      'user': USER_ID,
      'output': 'json',
      'input': 'json',
      'query': json.dumps(record_data, ensure_ascii=False)
    }
    
    try:
      response = requests.get(url, params=params, timeout=10)
      response.raise_for_status()
      data = response.json()
      
      if 'errors' in data:
        return None
      
      return data
    except Exception as e:
      return None

# ============================================================================
# ОБЩАЯ ФУНКЦИЯ (ДЛЯ ОДНОШАГОВОЙ ОБРАБОТКИ)
# ============================================================================

def process_excel():
    """
    Полная обработка: предобработка + API.
    """
    print("\n!!! ВНИМАНИЕ: Запущена полная обработка в два этапа !!!\n")
    
    # Этап 1: Предобработка
    df = preprocess_excel()
    if df is None:
        print("\n✗ Предобработка не завершена. Обработка остановлена.")
        return
    
    # Этап 2: API
    print("\n" + "="*80)
    print("ПЕРЕХОД К ОБРАБОТКЕ ЧЕРЕЗ API...")
    print("="*80)
    process_with_ahunter()


# ============================================================================
# ТЕСТИРОВАНИЕ
# ============================================================================

def test_single_row():
    """Тестирует предобработку на первой строке"""
    print("="*80)
    print("ТЕСТИРОВАНИЕ ПРЕДОБРАБОТКИ")
    print("="*80)
    
    try:
        df = pd.read_excel(INPUT_FILE)
    except Exception as e:
        print(f"✗ Ошибка: {e}")
        return
    
    if len(df) == 0:
        print("✗ Файл пустой!")
        return
    
    print(f"Загружено строк: {len(df)}")
    print("\n" + "="*80)
    print("ИСХОДНЫЕ ДАННЫЕ (первая строка)")
    print("="*80)
    
    row = df.iloc[0]
    for col in df.columns:
        print(f"{col}: {row[col]} (тип: {type(row[col]).__name__})")
    
    print("\n" + "="*80)
    print("ПОСЛЕ ПРЕДОБРАБОТКИ")
    print("="*80)
    
    # Тест очистки ФИО
    if 'ФИО' in df.columns:
        original = row['ФИО']
        cleaned = clean_fio(original)
        print(f"\nФИО:")
        print(f"  Было: {original}")
        print(f"  Стало: {cleaned}")
    
    # Тест очистки ИНН
    if 'ИНН' in df.columns:
        original = row['ИНН']
        cleaned = clean_inn(original)
        print(f"\nИНН:")
        print(f"  Было: {original} (тип: {type(original).__name__})")
        print(f"  Стало: {cleaned} (тип: {type(cleaned).__name__})")
    
    # Тест разделения
    print("\n" + "="*80)
    print("РАЗДЕЛЕНИЕ МНОЖЕСТВЕННЫХ ЗНАЧЕНИЙ")
    print("="*80)
    
    for col in ['ФИО', 'ТЕЛЕФОН', 'E-MAIL', 'ИНН']:
        if col in df.columns:
            original = row[col]
            if not pd.isna(original):
                if col == 'ИНН':
                    split_vals = split_multiple_values(original, 'inn')
                elif col == 'ФИО':
                    split_vals = split_multiple_values(original, 'fio_company')
                else:
                    split_vals = split_multiple_values(original, 'phone_email')
                
                if len(split_vals) > 1:
                    print(f"\n{col}:")
                    print(f"  Было: {original}")
                    for i, val in enumerate(split_vals, 1):
                        print(f"  Вариант {i}: {val}")
    
    print("\n" + "="*80)
    print("ТЕСТ ЗАВЕРШЕН")
    print("="*80)

In [9]:
# 1. Тест предобработки на одной строке
# test_single_row()

In [10]:
# 2. Предварительная обработка (создаст СВОД_preprocessed.xlsx)
preprocess_excel()

ПРЕДВАРИТЕЛЬНАЯ ОБРАБОТКА ДАННЫХ

1. Загрузка файла: СВОД.xlsx
✓ Загружено строк: 216
✓ Колонки: ['ФИО', 'ДАТА РОЖДЕНИЯ', 'ИНН', 'ТЕЛЕФОН', 'E-MAIL', 'АДРЕС']

2. Очистка данных...
  ✓ Очищены ФИО
  ✓ Очищены ИНН (убрано слово 'ИНН')

3. Разделение строк с множественными значениями...
  ✓ Было строк: 216, стало: 217

4. Локальная нормализация...
  ✓ Нормализованы email
  ✓ Нормализованы даты рождения
  ✓ Нормализованы ИНН (убраны пробелы)

5. Сохранение предварительной таблицы: СВОД_preprocessed.xlsx
✓ Готово!
  Строк в предварительной таблице: 217
  Файл сохранен: СВОД_preprocessed.xlsx

ПРЕДВАРИТЕЛЬНАЯ ОБРАБОТКА ЗАВЕРШЕНА

Проверьте файл СВОД_preprocessed.xlsx
Если всё корректно, запустите process_with_ahunter() для обработки через API


,ФИО,ДАТА РОЖДЕНИЯ,ИНН,ТЕЛЕФОН,E-MAIL,АДРЕС
0,Машков Василий Вячеславович,11.12.1979,,+7(921)047-55-55,mashkov_vasiliy@mail.ru,"183038, Россия, Мурманская обл, г Мурманск, ул..."
1,Машков Василий Вячеславович,11.12.1979,,+7(911)300-85-85,mashkov_vasiliy@mail.ru,"183038, Россия, Мурманская обл, г Мурманск, ул..."
2,"ООО ""ШАГИ""",,5190073518,+7(964)680-06-06,info@shagi.legal,"Мурманск, Новое плато, д.1"
3,NaN,,,NaN,client.law@mail.ru,NaN
4,"ООО ""РЕСПЕКТ""",,2901135536,NaN,NaN,NaN
...,...,...,...,...,...,...
211,Гомин Дмитрий Павлович,27.06.2003,,NaN,NaN,NaN
212,Сердюк Сергей Борисович,29.03.1986,,NaN,NaN,NaN
213,Баянова Виктория Васильевна,21.09.1976,,8-965-982-52-90,NaN,"г. Омск, пр-кт К.Маркса, 18/1, оф. 225"
214,Марков Леонтий Геннадьевич,25.12.1979,,8-913-616-31-48,NaN,"г. Омск, Герцена, д. 65а, оф. 2-12"


In [11]:
# 3. Обработка через API (загрузит СВОД_preprocessed.xlsx, создаст СВОД_cleaned.xlsx)
process_with_ahunter()

ОБРАБОТКА ЧЕРЕЗ AHUNTER API

1. Загрузка предварительной таблицы: СВОД_preprocessed.xlsx
✓ Загружено строк: 217
✓ Колонки: ['ФИО', 'ДАТА РОЖДЕНИЯ', 'ИНН', 'ТЕЛЕФОН', 'E-MAIL', 'АДРЕС']

2. Обработка через AHunter API...
 Обработка строки 1 из 217...
 Обработка строки 11 из 217...
 Обработка строки 21 из 217...
 Обработка строки 31 из 217...
 Обработка строки 41 из 217...
 Обработка строки 51 из 217...
 Обработка строки 61 из 217...
 Обработка строки 71 из 217...
 Обработка строки 81 из 217...
 Обработка строки 91 из 217...
 Обработка строки 101 из 217...
 Обработка строки 111 из 217...
 Обработка строки 121 из 217...
 Обработка строки 131 из 217...
 Обработка строки 141 из 217...
 Обработка строки 151 из 217...
 Обработка строки 161 из 217...
 Обработка строки 171 из 217...
 Обработка строки 181 из 217...
 Обработка строки 191 из 217...
 Обработка строки 201 из 217...
 Обработка строки 211 из 217...

3. Сохранение финальной таблицы: СВОД_cleaned.xlsx
✓ Готово!
 Всего строк: 217
 Успешн

In [7]:
# 4. Полная обработка в два этапа
# process_excel()